# Notebook 5 - Feature Engineering and Pre-Model Preparation

Purpose: Build model-ready datasets from the harmonised panel.  
Produces features for:
  - ARIMA: yearly aggregate time series (total complaints and total losses)
  - Random Forest: per-crime-type feature matrix with lag features

Output:  
  data/processed/arima_yearly_totals.csv  
  data/processed/rf_feature_matrix.csv

## 1. Imports and Load Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

panel = pd.read_csv('data/processed/ic3_panel_long_harmonised.csv')
panel.columns = [c.strip() for c in panel.columns]
panel = panel.sort_values(['Crime Type Harmonised', 'Year']).reset_index(drop=True)

print("Loaded:", panel.shape)
print("Years:", sorted(panel['Year'].unique()))
print("Crime types:", panel['Crime Type Harmonised'].nunique())

Loaded: (390, 4)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Crime types: 39


## 2. ARIMA Dataset - Yearly Aggregates

In [3]:
# Exclude discontinued categories from totals
DISCONTINUED = ['Auction', 'Criminal Forums', 'No Lead Value']

panel_active = panel[~panel['Crime Type Harmonised'].isin(DISCONTINUED)].copy()

arima_df = (
    panel_active
    .groupby('Year')[['Victim Count', 'Loss $']]
    .sum()
    .reset_index()
    .rename(columns={'Victim Count': 'Total Complaints', 'Loss $': 'Total Loss'})
)

arima_df['Total Loss (Billions USD)'] = (arima_df['Total Loss'] / 1e9).round(4)

print(arima_df.to_string(index=False))

arima_df.to_csv('data/processed/arima_yearly_totals.csv', index=False)
print("\nSaved: data/processed/arima_yearly_totals.csv")

 Year  Total Complaints   Total Loss  Total Loss (Billions USD)
 2015          335003.0 1.221199e+09                     1.2212
 2016          354400.0 1.443696e+09                     1.4437
 2017          334326.0 1.720592e+09                     1.7206
 2018          423743.0 3.415816e+09                     3.4158
 2019          501119.0 4.384107e+09                     4.3841
 2020          780403.0 5.046417e+09                     5.0464
 2021          775222.0 7.647476e+09                     7.6475
 2022          685738.0 1.093060e+10                    10.9306
 2023          691701.0 1.234724e+10                    12.3472
 2024          651865.0 1.608841e+10                    16.0884

Saved: data/processed/arima_yearly_totals.csv


In [4]:
# Structural Break (Post-2020)

arima_df['Post_2020'] = (arima_df['Year'] >= 2020).astype(int)

pre = arima_df[arima_df['Post_2020'] == 0]['Total Loss']
post = arima_df[arima_df['Post_2020'] == 1]['Total Loss']

print("Pre-2020 Avg Loss:", pre.mean())
print("Post-2020 Avg Loss:", post.mean())

Pre-2020 Avg Loss: 2437081841.8
Post-2020 Avg Loss: 10412028372.6


### Structural Break Insight

There is a clear increase in average financial losses after 2020 compared to the pre-2020 period.

This suggests a structural shift in cybercrime dynamics, likely driven by increased digital activity during the COVID-19 pandemic.

This supports the idea that cybercrime evolution is influenced by external socio-economic factors, not just technological change.

## 3. Check Stationarity - Required Before ARIMA

In [16]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name):
    result = adfuller(series.dropna())
    print(f"{name}")
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    if result[1] <= 0.05:
        print("  Result        : Stationary (p <= 0.05) - ARIMA can use as-is")
    else:
        print("  Result        : Non-stationary (p > 0.05) - needs differencing (d=1)")
    print()

adf_test(arima_df['Total Complaints'], 'Total Complaints')
adf_test(arima_df['Total Loss'],       'Total Loss ($)')

Total Complaints
  ADF Statistic : -1.1871
  p-value       : 0.6791
  Result        : Non-stationary (p > 0.05) - needs differencing (d=1)

Total Loss ($)
  ADF Statistic : 2.0139
  p-value       : 0.9987
  Result        : Non-stationary (p > 0.05) - needs differencing (d=1)



## 4. Random Forest Dataset - Per Crime Type Feature Matrix

In [17]:
# Only use active crime types with full or near-full data
min_years = 7
year_counts = panel_active.groupby('Crime Type Harmonised')['Year'].count()
eligible = year_counts[year_counts >= min_years].index.tolist()

panel_rf = panel_active[panel_active['Crime Type Harmonised'].isin(eligible)].copy()
panel_rf = panel_rf.sort_values(['Crime Type Harmonised', 'Year']).reset_index(drop=True)

print(f"Crime types with at least {min_years} years of data: {len(eligible)}")

Crime types with at least 7 years of data: 36


## 5. Build Lag Features

In [19]:
panel_rf = panel_rf.sort_values(['Crime Type Harmonised', 'Year'])

grp = panel_rf.groupby('Crime Type Harmonised')

# Lag features: previous 1 and 2 years
panel_rf['Complaints_Lag1']  = grp['Victim Count'].shift(1)
panel_rf['Complaints_Lag2']  = grp['Victim Count'].shift(2)
panel_rf['Loss_Lag1']        = grp['Loss $'].shift(1)
panel_rf['Loss_Lag2']        = grp['Loss $'].shift(2)

# Rolling mean over last 3 years
panel_rf['Complaints_Roll3'] = grp['Victim Count'].transform(lambda x: x.shift(1).rolling(3).mean())
panel_rf['Loss_Roll3']       = grp['Loss $'].transform(lambda x: x.shift(1).rolling(3).mean())

# YoY growth rates
panel_rf['Complaints_YoY']   = grp['Victim Count'].pct_change().replace([np.inf, -np.inf], np.nan)
panel_rf['Loss_YoY']         = grp['Loss $'].pct_change().replace([np.inf, -np.inf], np.nan)

# Loss per complaint ratio
panel_rf['Loss_per_Complaint'] = (
    panel_rf['Loss $'] / panel_rf['Victim Count'].replace(0, np.nan)
)

print("Columns:", list(panel_rf.columns))
print("Shape before dropping NaN:", panel_rf.shape)

Columns: ['Crime Type Harmonised', 'Year', 'Victim Count', 'Loss $', 'Complaints_Lag1', 'Complaints_Lag2', 'Loss_Lag1', 'Loss_Lag2', 'Complaints_Roll3', 'Loss_Roll3', 'Complaints_YoY', 'Loss_YoY', 'Loss_per_Complaint']
Shape before dropping NaN: (360, 13)


## 6. Encode Crime Type and Finalise Feature Matrix

In [21]:
# Encode crime type as integer category
panel_rf['Crime_Code'] = pd.Categorical(panel_rf['Crime Type Harmonised']).codes

# Define feature columns and targets
FEATURE_COLS = [
    'Year',
    'Crime_Code',
    'Complaints_Lag1',
    'Complaints_Lag2',
    'Complaints_Roll3',
    'Loss_Lag1',
    'Loss_Lag2',
    'Loss_Roll3',
    'Complaints_YoY',
    'Loss_YoY',
    'Loss_per_Complaint'
]

TARGET_COMPLAINTS = 'Victim Count'
TARGET_LOSS       = 'Loss $'

# Drop rows where any feature is NaN (lag rows from first 2 years)
rf_matrix = panel_rf[FEATURE_COLS + [TARGET_COMPLAINTS, TARGET_LOSS, 'Crime Type Harmonised']].dropna()

print("Final feature matrix shape:", rf_matrix.shape)
print("Years covered:", sorted(rf_matrix['Year'].unique()))
print("Crime types covered:", rf_matrix['Crime Type Harmonised'].nunique())
print()
print("Features:", FEATURE_COLS)
print("Targets :", TARGET_COMPLAINTS, "and", TARGET_LOSS)

Final feature matrix shape: (188, 14)
Years covered: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Crime types covered: 34

Features: ['Year', 'Crime_Code', 'Complaints_Lag1', 'Complaints_Lag2', 'Complaints_Roll3', 'Loss_Lag1', 'Loss_Lag2', 'Loss_Roll3', 'Complaints_YoY', 'Loss_YoY', 'Loss_per_Complaint']
Targets : Victim Count and Loss $


## 7. Train / Test Split

In [23]:
# Hold out final 2 years as test set (temporal split - never random split time series)
SPLIT_YEAR = 2022

train = rf_matrix[rf_matrix['Year'] <= SPLIT_YEAR].copy()
test  = rf_matrix[rf_matrix['Year'] >  SPLIT_YEAR].copy()

print(f"Train: {train.shape}  years {train['Year'].min()} to {train['Year'].max()}")
print(f"Test : {test.shape}   years {test['Year'].min()} to {test['Year'].max()}")

Train: (141, 14)  years 2018 to 2022
Test : (47, 14)   years 2023 to 2024


# Note
## Split is temporal (2023-2024 held out) not random. This prevents data leakage across time. 

## 8. Save Feature Matrix

In [25]:
rf_matrix.to_csv('data/processed/rf_feature_matrix.csv', index=False)
train.to_csv('data/processed/rf_train.csv', index=False)
test.to_csv('data/processed/rf_test.csv', index=False)

print("Saved:")
print("  data/processed/rf_feature_matrix.csv")
print("  data/processed/rf_train.csv")
print("  data/processed/rf_test.csv")

Saved:
  data/processed/rf_feature_matrix.csv
  data/processed/rf_train.csv
  data/processed/rf_test.csv
